In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, f1_score, classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek

# 1. Generate Synthetic Imbalanced Dataset (99% / 1% split)
X, y = make_classification(
    n_samples=50000, n_features=20, n_classes=2, 
    weights=[0.99, 0.01], flip_y=0, random_state=42
)

# Initialize Stratified K-Fold to maintain class ratios across splits
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Architect the Pipeline (Lever 1 + Lever 2 combined)
# We balance data via SMOTE-Tomek AND apply moderate class weights to the model
pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('resampler', SMOTETomek(random_state=42)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# 3. Generate Out-of-Fold Predictions safely without Data Leakage
print("--- Training Pipeline across CV Folds ---")
y_proba = cross_val_predict(pipeline, X, y, cv=cv, method='predict_proba')[:, 1]

# 4. Threshold Sweep / Post-Processing (Lever 3)
precisions, recalls, thresholds = precision_recall_curve(y, y_proba)

# Calculate F1 scores safely (adding epsilon to prevent division by zero)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores[:-1]) # Match dimensions with thresholds
best_threshold = thresholds[best_idx]

# 5. Evaluate the Operational Difference
y_pred_default = (y_proba >= 0.5).astype(int)
y_pred_optimal = (y_proba >= best_threshold).astype(int)

print(f"\nOptimal Operational Threshold: {best_threshold:.4f}\n")
print("=== Performance with Default Threshold (0.5) ===")
print(classification_report(y, y_pred_default, target_names=['Normal', 'Fraud']))

print("=== Performance with Optimal Threshold ===")
print(classification_report(y, y_pred_optimal, target_names=['Normal', 'Fraud']))

--- Training Pipeline across CV Folds ---

Optimal Operational Threshold: 0.9603

=== Performance with Default Threshold (0.5) ===
              precision    recall  f1-score   support

      Normal       1.00      0.96      0.98     49500
       Fraud       0.19      0.92      0.31       500

    accuracy                           0.96     50000
   macro avg       0.59      0.94      0.65     50000
weighted avg       0.99      0.96      0.97     50000

=== Performance with Optimal Threshold ===
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     49500
       Fraud       0.83      0.76      0.80       500

    accuracy                           1.00     50000
   macro avg       0.91      0.88      0.90     50000
weighted avg       1.00      1.00      1.00     50000

